# 02. Обучение и сохранение весов

Этот ноутбук читает признаки из `outputs/datasets`, затем последовательно показывает код direct ensemble, high-wind clip и two-stage correction. Веса сохраняются в `outputs/models/model_artifacts.joblib`, чтобы valid/test можно было пересчитать без повторного обучения.

In [1]:
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.interpolate import PchipInterpolator
from scipy.spatial.distance import pdist, squareform

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [2]:
# Управление отображением и диагностическими артефактами текущего запуска.
# Финальный submission сохраняется в каталоге текущего запуска.
PLOT_RESEARCH_OUTPUTS = True
PLOT_TWO_STAGE_DIAGNOSTICS = False
SAVE_DIAGNOSTIC_ARTIFACTS = False
SAVE_DIRECT_DEBUG_SUBMISSIONS = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [3]:
RANDOM_STATE = 42

In [4]:
# -------------------------
# Физика станции
# -------------------------
INSTALLED_CAPACITY_MW = 90.09
TURBINES_TOTAL = 26

CUT_IN_SPEED = 3.0
RATED_SPEED = 12.0
CUT_OUT_SPEED = 25.0

AIR_DENSITY_REF = 1.225
EPS = 1e-6


In [5]:
# -------------------------
# Пути проекта
# -------------------------
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
DATA_OUT_DIR = OUT_DIR / "data"
FIGURE_DIR = OUT_DIR / "figures"

for directory in [OUT_DIR, DATA_OUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CANDIDATES = [
    DATA_DIR / "train_merged.csv",
    DATA_DIR / "train.csv",
]

VALID_CANDIDATES = [
    DATA_DIR / "valid_merged.csv",
    DATA_DIR / "valid.csv",
]

TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if p.exists()), TRAIN_CANDIDATES[0])
VALID_PATH = next((p for p in VALID_CANDIDATES if p.exists()), VALID_CANDIDATES[0])

TURBINE_COORDS_CANDIDATES = [
    Path("map") / "data" / "wind_farm_coords.csv",
    Path("map") / "map_data" / "wind_farm_cords.csv",
    Path("map") / "wind_data" / "wind_farm_cords.csv",
]
TURBINE_COORDS_PATH = next((p for p in TURBINE_COORDS_CANDIDATES if p.exists()), TURBINE_COORDS_CANDIDATES[0])

# Основной файл для отправки.
SUB_PATH = OUT_DIR / "submission_final.csv"

# Диагностические submission-файлы direct ensemble.
DIRECT_SUB_PATH = DATA_OUT_DIR / "direct_full_clipped.csv"
DIRECT_RAW_PATH = DATA_OUT_DIR / "direct_full_raw.csv"

LOG_DIR = DATA_OUT_DIR
RESEARCH_PLOT_DIR = FIGURE_DIR
TS_DIR = DATA_OUT_DIR / "two_stage"


In [6]:
# -------------------------
# Блоки текущего запуска
# -------------------------
FULL_PHYSICS_BLOCK_ENABLED = True
HIGH_WIND_CLIP_ENABLED = True
TWO_STAGE_ENABLED = True

RUN_FINAL_PIPELINE = True
RUN_LOCAL_CHECK = False
PLOT_FINAL_DISTRIBUTIONS = True
RUN_POWER_CURVE_DIAGNOSTIC = True
RUN_HIGH_WIND_CLIP_TUNING = False


In [7]:
# -------------------------
# Веса direct ensemble
# Можно аккуратно крутить веса, но сумма должна быть около 1.
# -------------------------
BLEND_WEIGHTS = {
    "cat_mae_direct": 0.361831,
    "hgb_q545": 0.235409,
    "xgb_residual": 0.177644,
    "hgb_q570": 0.116581,
    "lgb_residual": 0.064663,
    "hgb_q530": 0.043873,
}

# Размер direct ensemble. Если нужно быстрее — уменьши значения или поставь FAST_MODE в отдельных местах.
ENSEMBLE_PARAMS_FULL = {
    "cat_iter": 1200,
    "xgb_estimators": 900,
    "lgb_estimators": 900,
    "hgb_iter": 650,
}

ENSEMBLE_PARAMS_FAST = {
    "cat_iter": 500,
    "xgb_estimators": 450,
    "lgb_estimators": 450,
    "hgb_iter": 350,
}

HGB_QUANTILES = [0.545, 0.570, 0.530]
DIRECT_ENSEMBLE_FAST_MODE = False


In [8]:
# -------------------------
# Геометрия / wake
# -------------------------
N_LAYOUT_CLUSTERS = 4
LAYOUT_KMEANS_N_INIT = 30

WAKE_DIRECTION_STEP_DEG = 5
WAKE_LATERAL_THRESHOLD_M = 260
WAKE_MAX_DOWNWIND_M = 2500
WAKE_DECAY_DOWNWIND_M = 800

BEST_WAKE_FEATURE = "layout_wake_risk_scalar_120m"


In [9]:
# -------------------------
# Физическая декомпозиция скрытых потерь
# -------------------------
FULL_MIN_IDEAL_MW = 3.0
FULL_K_HIDDEN_LOW = -0.25
FULL_K_HIDDEN_HIGH = 0.95
FULL_CHANGEPOINT_WINDOW = 72
FULL_CHANGEPOINT_MIN_DISTANCE = 96
FULL_CHANGEPOINT_THRESHOLD_MAD = 3.5
FULL_MAX_OFF_GRID = TURBINES_TOTAL
FULL_AVAIL_LAMBDA = 0.0015
FULL_UPWIND_WEIGHT_BOOST = 0.35
FULL_OOF_SPLITS = 5
FULL_RANDOM_STATE = RANDOM_STATE


In [10]:
# -------------------------
# Empirical high-wind clip
# Параметры high-wind clip зафиксированы в конфигурации запуска.
# -------------------------
HIGH_WIND_SPEED_COL = "wind_speed_120m"
HIGH_WIND_START_WS = 11.5
HIGH_WIND_TRANSITION = 0.45
HIGH_WIND_CAP_QUANTILE = 0.70
HIGH_WIND_CAP_BIN_WIDTH = 0.50
HIGH_WIND_CAP_MIN_COUNT = 5
HIGH_WIND_CAP_ROLLING_WINDOW = 3
HIGH_WIND_CAP_MARGIN_MW = 1.5
HIGH_WIND_HARD_MAX_CAP = 77.0
HIGH_WIND_CLIP_STRENGTH = 0.85
HIGH_WIND_MIN_PRED = 0.0
HIGH_WIND_MAX_PRED = INSTALLED_CAPACITY_MW

# Cap-кривая как входной сигнал до обучения direct ensemble.
# Target не клипуется: модель получает только оценку high-wind потолка и gate.
HIGH_WIND_CAP_FEATURES_ENABLED = True

# Финальная конфигурация post-processing для submission_final.csv.
FINAL_BENCHMARK_VARIANT = "submission_alpha_0p100_aggressive_q65_cap76"
FINAL_CLIP_CONFIG = {
    "name": "aggressive_q65_cap76",
    "quantile": 0.65,
    "margin_mw": 1.0,
    "hard_max_cap": 76.0,
    "strength": 1.00,
}


In [11]:
# ============================================================
# TWO-STAGE CONFIG: normal behavior + deviation
# ============================================================

TWO_STAGE_ENABLED = True
TWO_STAGE_FAST_MODE = False

# Количество time-series split для OOF normal/deviation
TWO_STAGE_N_SPLITS = 5

# Финальная смесь:
# final = (1 - alpha) * direct_full + alpha * two_stage
# Коэффициент смешивания задаёт долю two-stage поправки в финальном прогнозе.
FINAL_TWO_STAGE_ALPHA = 0.10

# ============================================================
# Residual / deviation target clipping
# ============================================================
# Квантильная обрезка оставляет residual-модели редкие режимы станции.
TWO_STAGE_RESIDUAL_Q_LOW = 0.005
TWO_STAGE_RESIDUAL_Q_HIGH = 0.995
TWO_STAGE_RESIDUAL_CLIP_MULT = 1.25

# ============================================================
# Deviation prediction safety
# ============================================================
# Не зажимаем deviation: residual-ветка должна иметь право на крупную поправку.
# Значение 999 фактически отключает дополнительный absolute clip,
# остаётся только квантильная обрезка target/prediction.
TWO_STAGE_DEVIATION_SHRINK = 1.00
TWO_STAGE_DEVIATION_ABS_CLIP_MW = 999.0

# ============================================================
# Bad OOF head handling
# ============================================================
# Ранний OOF-участок участвует в обучении residual-модели.
TWO_STAGE_DROP_BAD_OOF_HEAD = False
TWO_STAGE_BAD_HEAD_FRAC = 0.00

# ============================================================
# Physics gate
# ============================================================
# Gate равен 1.0: residual-поправка проходит без дополнительного приглушения.
TWO_STAGE_USE_PHYSICS_GATE = False
TWO_STAGE_GATE_MIN = 1.00
TWO_STAGE_GATE_MAX = 1.00

# Какие признаки убрать из normal-ветки, чтобы normal-модель была именно про нормальную генерацию,
# а не про скрытые потери/аномалии.
TWO_STAGE_DEVIATION_KEYWORDS = [
    "full_k_hidden",
    "full_hidden_loss",
    "full_p_reconstructed",
    "full_recon_minus",
    "full_k_meteo",
    "full_k_perf",
    "full_k_aging",
    "phi_ice",
    "phi_turbulence",
    "phi_yaw",
]

TS_DIR = OUT_DIR / "ts"
TS_PLOTS_DIR = TS_DIR / "plots"

display(Markdown(
    "### Контрольный снимок конфигурации\n\n"
    "Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска."
))

config_snapshot = pd.DataFrame([
    {"group": "paths", "parameter": "TRAIN_PATH", "value": str(TRAIN_PATH)},
    {"group": "paths", "parameter": "VALID_PATH", "value": str(VALID_PATH)},
    {"group": "paths", "parameter": "TURBINE_COORDS_PATH", "value": str(TURBINE_COORDS_PATH)},
    {"group": "paths", "parameter": "OUT_DIR", "value": str(OUT_DIR)},
    {"group": "paths", "parameter": "SUB_PATH", "value": str(SUB_PATH)},
    {"group": "paths", "parameter": "SAVE_DIAGNOSTIC_ARTIFACTS", "value": SAVE_DIAGNOSTIC_ARTIFACTS},
    {"group": "paths", "parameter": "SAVE_DIRECT_DEBUG_SUBMISSIONS", "value": SAVE_DIRECT_DEBUG_SUBMISSIONS},
    {"group": "direct", "parameter": "BLEND_WEIGHTS_SUM", "value": sum(BLEND_WEIGHTS.values())},
    {"group": "direct", "parameter": "DIRECT_ENSEMBLE_FAST_MODE", "value": DIRECT_ENSEMBLE_FAST_MODE},
    {"group": "physics", "parameter": "FULL_PHYSICS_BLOCK_ENABLED", "value": FULL_PHYSICS_BLOCK_ENABLED},
    {"group": "clip", "parameter": "HIGH_WIND_CLIP_ENABLED", "value": HIGH_WIND_CLIP_ENABLED},
    {"group": "clip", "parameter": "HIGH_WIND_SPEED_COL", "value": HIGH_WIND_SPEED_COL},
    {"group": "clip", "parameter": "HIGH_WIND_CAP_FEATURES_ENABLED", "value": HIGH_WIND_CAP_FEATURES_ENABLED},
    {"group": "two_stage", "parameter": "TWO_STAGE_ENABLED", "value": TWO_STAGE_ENABLED},
    {"group": "two_stage", "parameter": "FINAL_TWO_STAGE_ALPHA", "value": FINAL_TWO_STAGE_ALPHA},
    {"group": "final", "parameter": "FINAL_BENCHMARK_VARIANT", "value": FINAL_BENCHMARK_VARIANT},
    {"group": "final", "parameter": "FINAL_CLIP_CONFIG", "value": str(FINAL_CLIP_CONFIG)},
    {"group": "two_stage", "parameter": "TWO_STAGE_N_SPLITS", "value": TWO_STAGE_N_SPLITS},
    {"group": "two_stage", "parameter": "TWO_STAGE_USE_PHYSICS_GATE", "value": TWO_STAGE_USE_PHYSICS_GATE},
    {"group": "two_stage", "parameter": "TWO_STAGE_DROP_BAD_OOF_HEAD", "value": TWO_STAGE_DROP_BAD_OOF_HEAD},
    {"group": "two_stage", "parameter": "TWO_STAGE_RESIDUAL_Q", "value": f"{TWO_STAGE_RESIDUAL_Q_LOW} .. {TWO_STAGE_RESIDUAL_Q_HIGH}"},
])
display(config_snapshot)

### Контрольный снимок конфигурации

Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска.

,group,parameter,value
0,paths,TRAIN_PATH,data\train_merged.csv
1,paths,VALID_PATH,data\valid_merged.csv
2,paths,TURBINE_COORDS_PATH,map\data\wind_farm_coords.csv
3,paths,OUT_DIR,outputs
4,paths,SUB_PATH,outputs\submission_final.csv
5,paths,SAVE_DIAGNOSTIC_ARTIFACTS,False
6,paths,SAVE_DIRECT_DEBUG_SUBMISSIONS,False
7,direct,BLEND_WEIGHTS_SUM,1.000001
8,direct,DIRECT_ENSEMBLE_FAST_MODE,False
9,physics,FULL_PHYSICS_BLOCK_ENABLED,True


In [12]:
TARGET_CANDIDATES = [
    "Выработка. Результирующий расчет",
    "target",
    "Выработка",
]

DATETIME_CANDIDATES = [
    "METEOFORECASTHOUR_OPENM_Datetime",
    "Datetime",
    "datetime",
    "date",
    "time",
]

REPAIR_CANDIDATES = [
    "Кол-во_ВЭУ_в_ремонте",
    "turbines_in_repair",
    "repair",
]

In [13]:
# Для исследовательского вида оставляем локальные таблицы и ключевые графики в ноутбуке.
PLOT_RESEARCH_OUTPUTS = True
PLOT_FINAL_DISTRIBUTIONS = False
RUN_POWER_CURVE_DIAGNOSTIC = False
RUN_LOCAL_CHECK = False
SAVE_DIAGNOSTIC_ARTIFACTS = True
SAVE_DIRECT_DEBUG_SUBMISSIONS = True

DATASET_OUT_DIR = OUT_DIR / "datasets"
MODEL_OUT_DIR = OUT_DIR / "models"
for directory in [DATASET_OUT_DIR, MODEL_OUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [14]:
import json
import joblib
import ves_pipeline_core as vpc

feature_contract = vpc.validate_feature_contract()
train_model = pd.read_csv(vpc.TRAIN_FEATURES_PATH)
valid_model = pd.read_csv(vpc.VALID_FEATURES_PATH)
model_features = json.loads(vpc.MODEL_FEATURES_PATH.read_text(encoding="utf-8"))["model_features"]
display(Markdown("### Feature contract перед обучением"))
display(feature_contract)
display(pd.DataFrame([
    {"dataset": "train_model", "rows": len(train_model), "columns": train_model.shape[1]},
    {"dataset": "valid_model", "rows": len(valid_model), "columns": valid_model.shape[1]},
    {"dataset": "model_features", "rows": len(model_features), "columns": 1},
]))


### Feature contract перед обучением

,dataset,feature_rows,raw_rows,row_count_ok,missing_model_features
0,train,32434,32434,True,0
1,valid,2126,2126,True,0
2,test,1152,1152,True,0


,dataset,rows,columns
0,train_model,32434,213
1,valid_model,2126,206
2,model_features,185,1


## 1. Direct ensemble: модели, targets и blend

In [15]:

# ============================================================
# Ансамбль моделей — параметры заданы в глобальных настройках
# ============================================================

print("BLEND_WEIGHTS sum:", sum(BLEND_WEIGHTS.values()))
print("ENSEMBLE_PARAMS_FULL:", ENSEMBLE_PARAMS_FULL)
print("HGB_QUANTILES:", HGB_QUANTILES)


BLEND_WEIGHTS sum: 1.000001
ENSEMBLE_PARAMS_FULL: {'cat_iter': 1200, 'xgb_estimators': 900, 'lgb_estimators': 900, 'hgb_iter': 650}
HGB_QUANTILES: [0.545, 0.57, 0.53]


In [16]:
def fit_ensemble(local_train, feature_cols, label="ensemble", fast_mode=False):
    print(f"\nОбучение ансамбля: {label}")

    if "p_empirical_mean_80_120" not in local_train.columns:
        raise KeyError("В local_train нет p_empirical_mean_80_120")

    models = {}

    y_direct = local_train["target"].clip(0, INSTALLED_CAPACITY_MW)
    y_residual = local_train["target"] - local_train["p_empirical_mean_80_120"]
    y_scaled = (local_train["target"] / INSTALLED_CAPACITY_MW).clip(0, 1)

    params = ENSEMBLE_PARAMS_FAST if fast_mode else ENSEMBLE_PARAMS_FULL

    cat = CatBoostRegressor(
        iterations=params["cat_iter"],
        learning_rate=0.03,
        depth=6,
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
    )
    cat.fit(local_train[feature_cols], y_direct)
    models["cat_mae_direct"] = cat

    xgb = XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=params["xgb_estimators"],
        learning_rate=0.025,
        max_depth=5,
        min_child_weight=20,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        tree_method="hist",
        random_state=RANDOM_STATE,
    )
    xgb.fit(local_train[feature_cols], y_residual)
    models["xgb_residual"] = xgb

    lgbm = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=params["lgb_estimators"],
        learning_rate=0.025,
        num_leaves=31,
        min_child_samples=35,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    lgbm.fit(local_train[feature_cols], y_residual)
    models["lgb_residual"] = lgbm

    for q in HGB_QUANTILES:
        model_name = f"hgb_q{int(q * 1000):03d}"

        hgb = HistGradientBoostingRegressor(
            loss="quantile",
            quantile=q,
            max_iter=params["hgb_iter"],
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=0.02,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=RANDOM_STATE,
        )

        hgb.fit(local_train[feature_cols], y_scaled)
        models[model_name] = hgb

    return models


def predict_ensemble(models, frame, feature_cols):
    if "p_empirical_mean_80_120" not in frame.columns:
        raise KeyError("В frame нет p_empirical_mean_80_120")

    preds = {}

    preds["cat_mae_direct"] = np.clip(
        models["cat_mae_direct"].predict(frame[feature_cols]),
        0,
        INSTALLED_CAPACITY_MW,
    )

    preds["xgb_residual"] = np.clip(
        frame["p_empirical_mean_80_120"].to_numpy()
        + models["xgb_residual"].predict(frame[feature_cols]),
        0,
        INSTALLED_CAPACITY_MW,
    )

    preds["lgb_residual"] = np.clip(
        frame["p_empirical_mean_80_120"].to_numpy()
        + models["lgb_residual"].predict(frame[feature_cols]),
        0,
        INSTALLED_CAPACITY_MW,
    )

    for q in HGB_QUANTILES:
        model_name = f"hgb_q{int(q * 1000):03d}"

        preds[model_name] = (
            np.clip(models[model_name].predict(frame[feature_cols]), 0, 1)
            * INSTALLED_CAPACITY_MW
        )

    final = np.zeros(len(frame))

    for model_name, weight in BLEND_WEIGHTS.items():
        final += preds[model_name] * weight

    final = np.clip(final, 0, INSTALLED_CAPACITY_MW)

    return final, preds

## 2. High-wind clip для стабильного постпроцессинга

In [17]:
# ============================================================
# EMPIRICAL HIGH-WIND CLIP
# ============================================================

# Параметры high-wind clip заданы в секции глобальных настроек.
print("HIGH_WIND_CLIP_ENABLED:", HIGH_WIND_CLIP_ENABLED)
print("HIGH_WIND_SPEED_COL:", HIGH_WIND_SPEED_COL)
print("HIGH_WIND_START_WS:", HIGH_WIND_START_WS)
print("HIGH_WIND_CAP_QUANTILE:", HIGH_WIND_CAP_QUANTILE)
print("HIGH_WIND_CLIP_STRENGTH:", HIGH_WIND_CLIP_STRENGTH)


HIGH_WIND_CLIP_ENABLED: True
HIGH_WIND_SPEED_COL: wind_speed_120m
HIGH_WIND_START_WS: 11.5
HIGH_WIND_CAP_QUANTILE: 0.7
HIGH_WIND_CLIP_STRENGTH: 0.85


In [18]:
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


In [19]:
def fit_high_wind_cap_curve(
    reference,
    speed_col=HIGH_WIND_SPEED_COL,
    target_col="target",
    start_ws=HIGH_WIND_START_WS,
    bin_width=HIGH_WIND_CAP_BIN_WIDTH,
    min_count=HIGH_WIND_CAP_MIN_COUNT,
    quantile=HIGH_WIND_CAP_QUANTILE,
    rolling_window=HIGH_WIND_CAP_ROLLING_WINDOW,
    margin_mw=HIGH_WIND_CAP_MARGIN_MW,
    hard_max_cap=HIGH_WIND_HARD_MAX_CAP,
):
    """
    Строит empirical high-wind cap по фактической выработке train.

    Особенность этой эмпирической cap-кривой:
    - берём только строки с wind >= start_ws;
    - cap НЕ обязан быть монотонным;
    - cap ограничивается hard_max_cap, чтобы не улетать к 80+ МВт;
    - итоговая линия получается ближе к реальному потолку факта при сильном ветре.
    """
    if speed_col not in reference.columns:
        raise ValueError(f"Нет speed_col: {speed_col}")

    if target_col not in reference.columns:
        raise ValueError(f"Нет target_col: {target_col}")

    ref = reference[[speed_col, target_col]].copy()
    ref[speed_col] = pd.to_numeric(ref[speed_col], errors="coerce")
    ref[target_col] = pd.to_numeric(ref[target_col], errors="coerce")
    ref = ref.replace([np.inf, -np.inf], np.nan).dropna()

    if len(ref) == 0:
        raise ValueError("Пустой reference для high-wind cap curve.")

    high_ref = ref[ref[speed_col] >= start_ws].copy()

    # Защита на случай, если в конкретном split мало high-wind точек.
    if len(high_ref) < max(20, min_count * 2):
        high_ref = ref.copy()

    high_ref["ws_bin"] = (high_ref[speed_col] / bin_width).round() * bin_width

    cap_curve = (
        high_ref
        .groupby("ws_bin", observed=True)
        .agg(
            count=(target_col, "size"),
            speed_mean=(speed_col, "mean"),
            target_mean=(target_col, "mean"),
            target_median=(target_col, "median"),
            target_q=(target_col, lambda x: np.quantile(x, quantile)),
            target_max=(target_col, "max"),
        )
        .reset_index()
        .sort_values("speed_mean")
        .reset_index(drop=True)
    )

    cap_curve = cap_curve[cap_curve["count"] >= min_count].copy()
    cap_curve = cap_curve.sort_values("speed_mean").reset_index(drop=True)

    if len(cap_curve) < 2:
        fallback_cap = float(np.nanquantile(high_ref[target_col], quantile) + margin_mw)
        fallback_cap = float(np.clip(fallback_cap, 0, min(hard_max_cap, INSTALLED_CAPACITY_MW)))
        max_speed = max(float(high_ref[speed_col].max()), start_ws + 1.0)
        cap_curve = pd.DataFrame({
            "ws_bin": [start_ws, max_speed],
            "count": [len(high_ref), len(high_ref)],
            "speed_mean": [start_ws, max_speed],
            "target_mean": [fallback_cap, fallback_cap],
            "target_median": [fallback_cap, fallback_cap],
            "target_q": [fallback_cap, fallback_cap],
            "target_max": [fallback_cap, fallback_cap],
        })

    cap_curve["cap_raw"] = cap_curve["target_q"] + margin_mw

    # Сглаживаем, чтобы cap не прыгал от бина к бину.
    if rolling_window and rolling_window > 1 and len(cap_curve) >= rolling_window:
        cap_curve["cap_smooth"] = (
            cap_curve["cap_raw"]
            .rolling(rolling_window, center=True, min_periods=1)
            .median()
        )
    else:
        cap_curve["cap_smooth"] = cap_curve["cap_raw"]

    # Главная защита: high-wind cap не должен быть выше реалистичного потолка.
    cap_curve["cap_final"] = np.minimum(cap_curve["cap_smooth"], hard_max_cap)
    cap_curve["cap_final"] = cap_curve["cap_final"].clip(0, INSTALLED_CAPACITY_MW)

    # Служебные параметры, чтобы потом в csv было видно, как строилась таблица.
    cap_curve["start_ws"] = start_ws
    cap_curve["quantile"] = quantile
    cap_curve["margin_mw"] = margin_mw
    cap_curve["hard_max_cap"] = hard_max_cap
    cap_curve["bin_width"] = bin_width

    return cap_curve


In [20]:
def apply_high_wind_smart_clip(
    frame,
    pred_values,
    cap_curve,
    speed_col=HIGH_WIND_SPEED_COL,
    start_ws=HIGH_WIND_START_WS,
    transition=HIGH_WIND_TRANSITION,
    strength=HIGH_WIND_CLIP_STRENGTH,
    min_pred=HIGH_WIND_MIN_PRED,
    max_pred=HIGH_WIND_MAX_PRED,
):
    """
    Мягкий high-wind clip:
    corrected = pred - strength * gate(high wind) * max(pred - empirical_cap, 0)

    empirical_cap берётся из train-факта по high-wind бинам.
    """
    if speed_col not in frame.columns:
        raise ValueError(f"Нет speed_col: {speed_col}")

    if "cap_final" not in cap_curve.columns:
        raise ValueError("В cap_curve нет колонки cap_final")

    pred = np.asarray(pred_values, dtype=float)
    ws = pd.to_numeric(frame[speed_col], errors="coerce").to_numpy(dtype=float)

    curve_x = cap_curve["speed_mean"].to_numpy(dtype=float)
    curve_y = cap_curve["cap_final"].to_numpy(dtype=float)

    order = np.argsort(curve_x)
    curve_x = curve_x[order]
    curve_y = curve_y[order]

    ws_safe = np.nan_to_num(ws, nan=np.nanmedian(curve_x))

    cap = np.interp(
        ws_safe,
        curve_x,
        curve_y,
        left=curve_y[0],
        right=curve_y[-1],
    )

    cap = np.clip(cap, min_pred, max_pred)

    gate = sigmoid_np((ws_safe - start_ws) / transition)
    gate = np.nan_to_num(gate, nan=0.0)

    excess = np.maximum(pred - cap, 0.0)
    corrected = pred - strength * gate * excess
    corrected = np.clip(corrected, min_pred, max_pred)

    diagnostics = pd.DataFrame({
        "high_wind_speed": ws,
        "pred_raw": pred,
        "high_wind_cap_final": cap,
        "high_wind_gate": gate,
        "high_wind_excess": excess,
        "pred_clipped": corrected,
        "clip_delta": corrected - pred,
    })

    return corrected, diagnostics


## 3. Two-stage correction: normal behavior + residual/deviation

In [21]:
# ============================================================
# PHYSICAL DECOMPOSITION BLOCK
# P_ideal_clean -> K_hidden -> K_avail + K_perf
# ============================================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, mean_squared_error

# Параметры блока задаются в секции глобальных настроек.
print("FULL_PHYSICS_BLOCK_ENABLED:", FULL_PHYSICS_BLOCK_ENABLED)
print("FULL_OOF_SPLITS:", FULL_OOF_SPLITS)
print("FULL_CHANGEPOINT_WINDOW:", FULL_CHANGEPOINT_WINDOW)
print("FULL_AVAIL_LAMBDA:", FULL_AVAIL_LAMBDA)


FULL_PHYSICS_BLOCK_ENABLED: True
FULL_OOF_SPLITS: 5
FULL_CHANGEPOINT_WINDOW: 72
FULL_AVAIL_LAMBDA: 0.0015


In [22]:

# ============================================================
# TWO-STAGE DEVIATION CONFIG
# ============================================================

# Параметры two-stage correction заданы в секции глобальных настроек.
print("TWO_STAGE_ENABLED:", TWO_STAGE_ENABLED)
print("TWO_STAGE_FAST_MODE:", TWO_STAGE_FAST_MODE)
print("TWO_STAGE_N_SPLITS:", TWO_STAGE_N_SPLITS)
print("FINAL_TWO_STAGE_ALPHA:", FINAL_TWO_STAGE_ALPHA)
print("TS_DIR:", TS_DIR)


TWO_STAGE_ENABLED: True
TWO_STAGE_FAST_MODE: False
TWO_STAGE_N_SPLITS: 5
FINAL_TWO_STAGE_ALPHA: 0.1
TS_DIR: outputs\ts


In [23]:
# ============================================================
# TWO-STAGE HELPERS
# ============================================================


In [24]:
def _ts_existing_cols(train_df, valid_df, cols):
    return list(dict.fromkeys([
        c for c in cols
        if c in train_df.columns and c in valid_df.columns
    ]))


In [25]:
def _ts_forbidden_feature(c):
    c_low = str(c).lower()
    if c_low in {"target", "row_id", "source"}:
        return True
    if c.endswith("_train_only"):
        return True
    if "выработка" in c_low:
        return True
    return False


In [26]:
def _ts_clean_feature_list(train_df, valid_df, cols):
    cols = _ts_existing_cols(train_df, valid_df, cols)
    return [c for c in cols if not _ts_forbidden_feature(c)]


In [27]:
def _ts_prepare_matrix(train_df, valid_df, cols):
    cols = _ts_clean_feature_list(train_df, valid_df, cols)

    x_train = train_df[cols].copy()
    x_valid = valid_df[cols].copy()

    x_train = x_train.replace([np.inf, -np.inf], np.nan)
    x_valid = x_valid.replace([np.inf, -np.inf], np.nan)

    med = x_train.median(numeric_only=True)
    x_train = x_train.fillna(med).fillna(0)
    x_valid = x_valid.fillna(med).fillna(0)

    return x_train, x_valid, cols


In [28]:
def _ts_save_submission(valid_frame, pred, path):
    pred = np.asarray(pred, dtype=float)
    pred = np.clip(pred, 0, INSTALLED_CAPACITY_MW)

    if "row_id" not in valid_frame.columns:
        raise KeyError("В valid_frame нет row_id, не могу восстановить порядок submission.")

    sub = valid_frame[["row_id"]].copy()
    sub["target"] = pred
    sub = sub.sort_values("row_id")[["target"]].reset_index(drop=True)
    sub.to_csv(path, index=False)
    print("Saved:", path, "shape:", sub.shape)
    return sub


In [29]:
def _ts_apply_clip(train_frame, valid_frame, pred_raw, suffix):
    pred_raw = np.clip(np.asarray(pred_raw, dtype=float), 0, INSTALLED_CAPACITY_MW)

    if "HIGH_WIND_CLIP_ENABLED" not in globals() or not HIGH_WIND_CLIP_ENABLED:
        diag = pd.DataFrame({
            "pred_raw": pred_raw,
            "pred_clipped": pred_raw,
            "clip_delta": np.zeros(len(pred_raw)),
        })
        return pred_raw, diag

    cap_curve = fit_high_wind_cap_curve(
        train_frame,
        speed_col=HIGH_WIND_SPEED_COL,
        target_col="target",
    )

    pred_clip, clip_diag = apply_high_wind_smart_clip(
        valid_frame,
        pred_raw,
        cap_curve,
        speed_col=HIGH_WIND_SPEED_COL,
    )

    if SAVE_DIAGNOSTIC_ARTIFACTS:
        TS_DIR.mkdir(parents=True, exist_ok=True)
        cap_curve.to_csv(TS_DIR / f"high_wind_cap_curve_{suffix}.csv", index=False)
        clip_diag.to_csv(TS_DIR / f"high_wind_clip_diag_{suffix}.csv", index=False)

    return np.clip(pred_clip, 0, INSTALLED_CAPACITY_MW), clip_diag


In [30]:
def _ts_hgb_l1(seed, fast=False):
    return HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.040 if not fast else 0.060,
        max_iter=520 if not fast else 220,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.06,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=seed,
    )


In [31]:
def _ts_hgb_l2(seed, fast=False):
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.040 if not fast else 0.060,
        max_iter=520 if not fast else 220,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.04,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=seed,
    )


In [32]:
def _ts_hgb_q50(seed, fast=False):
    return HistGradientBoostingRegressor(
        loss="quantile",
        quantile=0.50,
        learning_rate=0.040 if not fast else 0.060,
        max_iter=500 if not fast else 220,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.06,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=seed,
    )


In [33]:
def _ts_cat_mae(seed, fast=False):
    return CatBoostRegressor(
        iterations=650 if not fast else 250,
        learning_rate=0.035 if not fast else 0.055,
        depth=5,
        loss_function="MAE",
        random_seed=seed,
        verbose=0,
        allow_writing_files=False,
    )


In [34]:
def _ts_lgb_l1(seed, fast=False):
    return lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=650 if not fast else 250,
        learning_rate=0.030 if not fast else 0.050,
        num_leaves=31,
        min_child_samples=35,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=1.5,
        random_state=seed,
        verbosity=-1,
    )


In [35]:
def _ts_xgb_l1(seed, fast=False):
    return XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=650 if not fast else 250,
        learning_rate=0.030 if not fast else 0.050,
        max_depth=5,
        min_child_weight=20,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=2.5,
        tree_method="hist",
        random_state=seed,
    )


In [36]:
def _ts_model_bank(kind, fast=False):
    if kind == "normal":
        return {
            "normal_hgb_l1": lambda seed: _ts_hgb_l1(seed, fast),
            "normal_hgb_l2": lambda seed: _ts_hgb_l2(seed, fast),
            "normal_hgb_q50": lambda seed: _ts_hgb_q50(seed, fast),
            "normal_cat_mae": lambda seed: _ts_cat_mae(seed, fast),
        }

    if kind == "deviation":
        return {
            "dev_hgb_l1": lambda seed: _ts_hgb_l1(seed, fast),
            "dev_hgb_q50": lambda seed: _ts_hgb_q50(seed, fast),
            "dev_lgb_l1": lambda seed: _ts_lgb_l1(seed, fast),
            "dev_xgb_l1": lambda seed: _ts_xgb_l1(seed, fast),
            "dev_cat_mae": lambda seed: _ts_cat_mae(seed, fast),
        }

    raise ValueError(f"unknown kind: {kind}")


In [37]:
def _ts_oof_and_valid_predictions(
    train_frame,
    valid_frame,
    feature_cols,
    y,
    model_bank,
    n_splits=5,
    clip_pred=None,
    label="stage",
):
    x_train, x_valid, used_cols = _ts_prepare_matrix(train_frame, valid_frame, feature_cols)
    y = np.asarray(y, dtype=float)

    tscv = TimeSeriesSplit(n_splits=n_splits, gap=24)

    oof_components = {}
    valid_components = {}
    fold_rows = []

    for model_i, (model_name, factory) in enumerate(model_bank.items(), 1):
        print("=" * 100)
        print(label, "model:", model_name)

        oof = np.full(len(train_frame), np.nan, dtype=float)
        valid_fold_preds = []

        for fold, (tr_idx, va_idx) in enumerate(tscv.split(x_train), 1):
            model = factory(RANDOM_STATE + 1000 * model_i + fold)
            model.fit(x_train.iloc[tr_idx], y[tr_idx])

            fold_pred = np.asarray(model.predict(x_train.iloc[va_idx]), dtype=float)
            valid_pred = np.asarray(model.predict(x_valid), dtype=float)

            if clip_pred is not None:
                fold_pred = np.clip(fold_pred, clip_pred[0], clip_pred[1])
                valid_pred = np.clip(valid_pred, clip_pred[0], clip_pred[1])

            oof[va_idx] = fold_pred
            valid_fold_preds.append(valid_pred)

            fold_mae = mean_absolute_error(y[va_idx], fold_pred)
            fold_rows.append({
                "stage": label,
                "model": model_name,
                "fold": fold,
                "mae": fold_mae,
                "n_train": len(tr_idx),
                "n_valid": len(va_idx),
            })

            print(f"fold {fold}: MAE={fold_mae:.6f}, n_train={len(tr_idx)}, n_valid={len(va_idx)}")

        if np.isnan(oof).any():
            fill_value = float(np.nanmedian(oof))
            oof = np.where(np.isnan(oof), fill_value, oof)
            print("OOF head rows filled by median:", fill_value)

        oof_components[model_name] = oof
        valid_components[model_name] = np.mean(valid_fold_preds, axis=0)

    oof_df = pd.DataFrame(oof_components)
    valid_df = pd.DataFrame(valid_components)

    oof_mean = oof_df.mean(axis=1).to_numpy()
    valid_mean = valid_df.mean(axis=1).to_numpy()

    if clip_pred is not None:
        oof_mean = np.clip(oof_mean, clip_pred[0], clip_pred[1])
        valid_mean = np.clip(valid_mean, clip_pred[0], clip_pred[1])

    fold_report = pd.DataFrame(fold_rows)
    return oof_mean, valid_mean, oof_df, valid_df, fold_report, used_cols


## 4. Обучение по сохраненным признакам и запись весов

In [38]:
# Sequential notebook step; kept inline instead of a one-shot wrapper.
final_models = fit_ensemble(
    train_model,
    model_features,
    label="saved_direct_ensemble",
    fast_mode=DIRECT_ENSEMBLE_FAST_MODE,
)
direct_valid_raw, direct_valid_components = predict_ensemble(final_models, valid_model, model_features)

# Sequential notebook step; kept inline instead of a one-shot wrapper.
direct_cap_curve = fit_high_wind_cap_curve(
    train_model,
    speed_col=HIGH_WIND_SPEED_COL,
    target_col="target",
)
direct_valid_clip, direct_clip_diag = apply_high_wind_smart_clip(
    valid_model,
    direct_valid_raw,
    direct_cap_curve,
    speed_col=HIGH_WIND_SPEED_COL,
)
direct_valid_clip = np.clip(direct_valid_clip, 0, INSTALLED_CAPACITY_MW)

# Sequential notebook step; kept inline instead of a one-shot wrapper.
ts_train = train_model.copy().reset_index(drop=True)
ts_valid = valid_model.copy().reset_index(drop=True)
if "datetime" in ts_train.columns:
    ts_train = ts_train.sort_values("datetime").reset_index(drop=True)

base_ts_cols = _ts_clean_feature_list(ts_train, ts_valid, model_features)
normal_feature_cols = [
    c for c in base_ts_cols
    if not any(key in c for key in TWO_STAGE_DEVIATION_KEYWORDS)
]
if len(normal_feature_cols) < 20:
    normal_feature_cols = base_ts_cols.copy()
deviation_feature_cols = base_ts_cols.copy()

y_train = ts_train["target"].clip(0, INSTALLED_CAPACITY_MW).to_numpy(dtype=float)
(
    normal_oof_pred,
    normal_valid_pred,
    normal_oof_components,
    normal_valid_components,
    normal_fold_report,
    normal_used_cols,
) = _ts_oof_and_valid_predictions(
    ts_train,
    ts_valid,
    normal_feature_cols,
    y_train,
    _ts_model_bank("normal", fast=TWO_STAGE_FAST_MODE),
    n_splits=TWO_STAGE_N_SPLITS,
    clip_pred=(0.0, INSTALLED_CAPACITY_MW),
    label="normal",
)
ts_train["two_stage_normal_pred"] = np.clip(normal_oof_pred, 0, INSTALLED_CAPACITY_MW)
ts_valid["two_stage_normal_pred"] = np.clip(normal_valid_pred, 0, INSTALLED_CAPACITY_MW)

ts_train["two_stage_deviation_target_raw"] = y_train - ts_train["two_stage_normal_pred"]
dev_q_low = float(ts_train["two_stage_deviation_target_raw"].quantile(TWO_STAGE_RESIDUAL_Q_LOW))
dev_q_high = float(ts_train["two_stage_deviation_target_raw"].quantile(TWO_STAGE_RESIDUAL_Q_HIGH))
dev_clip_low = dev_q_low * TWO_STAGE_RESIDUAL_CLIP_MULT
dev_clip_high = dev_q_high * TWO_STAGE_RESIDUAL_CLIP_MULT
if dev_clip_low > dev_clip_high:
    dev_clip_low, dev_clip_high = dev_clip_high, dev_clip_low

ts_train["two_stage_deviation_target"] = ts_train["two_stage_deviation_target_raw"].clip(dev_clip_low, dev_clip_high)
residual_train_mask = np.isfinite(ts_train["two_stage_deviation_target"].to_numpy(dtype=float))

# Sequential notebook step; kept inline instead of a one-shot wrapper.
for frame in [ts_train, ts_valid]:
    if "p_empirical_mean_80_120" in frame.columns:
        frame["two_stage_normal_minus_empirical"] = frame["two_stage_normal_pred"] - frame["p_empirical_mean_80_120"]
        frame["two_stage_normal_div_empirical"] = frame["two_stage_normal_pred"] / (frame["p_empirical_mean_80_120"].abs() + EPS)
    if "p_theory_mean_80_120" in frame.columns:
        frame["two_stage_normal_minus_theory"] = frame["two_stage_normal_pred"] - frame["p_theory_mean_80_120"]
        frame["two_stage_normal_div_theory"] = frame["two_stage_normal_pred"] / (frame["p_theory_mean_80_120"].abs() + EPS)
    if "full_p_ideal_clean" in frame.columns:
        frame["two_stage_normal_minus_ideal_clean"] = frame["two_stage_normal_pred"] - frame["full_p_ideal_clean"]
        frame["two_stage_normal_div_ideal_clean"] = frame["two_stage_normal_pred"] / (frame["full_p_ideal_clean"].abs() + EPS)
    if "full_hidden_loss_mw_pred" in frame.columns:
        frame["two_stage_normal_x_hidden_loss"] = frame["two_stage_normal_pred"] * frame["full_hidden_loss_mw_pred"] / INSTALLED_CAPACITY_MW
    if "layout_wake_risk_scalar_120m" in frame.columns:
        frame["two_stage_normal_x_wake_risk"] = frame["two_stage_normal_pred"] * frame["layout_wake_risk_scalar_120m"]

two_stage_meta_cols = [
    "two_stage_normal_pred",
    "two_stage_normal_minus_empirical",
    "two_stage_normal_div_empirical",
    "two_stage_normal_minus_theory",
    "two_stage_normal_div_theory",
    "two_stage_normal_minus_ideal_clean",
    "two_stage_normal_div_ideal_clean",
    "two_stage_normal_x_hidden_loss",
    "two_stage_normal_x_wake_risk",
]
deviation_feature_cols = _ts_clean_feature_list(ts_train, ts_valid, deviation_feature_cols + two_stage_meta_cols)

dev_train_df = ts_train.loc[residual_train_mask].copy().reset_index(drop=True)
y_dev = dev_train_df["two_stage_deviation_target"].to_numpy(dtype=float)
(
    dev_oof_part,
    dev_valid_pred_raw,
    dev_oof_components_part,
    dev_valid_components,
    dev_fold_report,
    dev_used_cols,
) = _ts_oof_and_valid_predictions(
    dev_train_df,
    ts_valid,
    deviation_feature_cols,
    y_dev,
    _ts_model_bank("deviation", fast=TWO_STAGE_FAST_MODE),
    n_splits=TWO_STAGE_N_SPLITS,
    clip_pred=(dev_clip_low, dev_clip_high),
    label="deviation",
)

# Sequential notebook step; kept inline instead of a one-shot wrapper.
ts_valid["two_stage_deviation_pred_raw"] = np.asarray(dev_valid_pred_raw, dtype=float)
dev_safe = np.clip(
    ts_valid["two_stage_deviation_pred_raw"].to_numpy(dtype=float),
    -TWO_STAGE_DEVIATION_ABS_CLIP_MW,
    TWO_STAGE_DEVIATION_ABS_CLIP_MW,
) * TWO_STAGE_DEVIATION_SHRINK
ts_valid["two_stage_physics_gate"] = 1.0
ts_valid["two_stage_deviation_pred_final"] = dev_safe * ts_valid["two_stage_physics_gate"].to_numpy(dtype=float)
ts_valid["two_stage_pred_raw"] = (
    ts_valid["two_stage_normal_pred"] + ts_valid["two_stage_deviation_pred_final"]
).clip(0, INSTALLED_CAPACITY_MW)

two_stage_valid_clip, two_stage_clip_diag = _ts_apply_clip(
    ts_train,
    ts_valid,
    ts_valid["two_stage_pred_raw"].to_numpy(dtype=float),
    suffix="two_stage_training",
)

# Sequential notebook step; kept inline instead of a one-shot wrapper.
alpha = float(np.clip(FINAL_TWO_STAGE_ALPHA, 0.0, 1.0))
blend_valid = np.clip((1.0 - alpha) * direct_valid_clip + alpha * two_stage_valid_clip, 0, INSTALLED_CAPACITY_MW)
final_clip_cfg = dict(FINAL_CLIP_CONFIG)
final_cap_curve = fit_high_wind_cap_curve(
    ts_train,
    speed_col=HIGH_WIND_SPEED_COL,
    target_col="target",
    quantile=final_clip_cfg["quantile"],
    margin_mw=final_clip_cfg["margin_mw"],
    hard_max_cap=final_clip_cfg["hard_max_cap"],
)
final_valid_pred, final_clip_diag = apply_high_wind_smart_clip(
    ts_valid,
    blend_valid,
    final_cap_curve,
    speed_col=HIGH_WIND_SPEED_COL,
    strength=final_clip_cfg["strength"],
)
final_valid_pred = np.clip(final_valid_pred, 0, INSTALLED_CAPACITY_MW)

# Sequential notebook step; kept inline instead of a one-shot wrapper.
normal_bank = vpc._train_model_bank(
    vpc._prepare_training_env(),
    ts_train,
    normal_feature_cols,
    y_train,
    "normal",
    clip_pred=(0.0, INSTALLED_CAPACITY_MW),
)
deviation_bank = vpc._train_model_bank(
    vpc._prepare_training_env(),
    dev_train_df,
    deviation_feature_cols,
    y_dev,
    "deviation",
    clip_pred=(dev_clip_low, dev_clip_high),
)

vpc.MODEL_DIR.mkdir(parents=True, exist_ok=True)
valid_training_pred = valid_model[["row_id"]].copy()
if "datetime" in valid_model.columns:
    valid_training_pred["datetime"] = pd.to_datetime(valid_model["datetime"], errors="coerce")
valid_training_pred["direct_pred_raw"] = direct_valid_raw
valid_training_pred["direct_pred_clip"] = direct_valid_clip
valid_training_pred["two_stage_pred_clip"] = two_stage_valid_clip
valid_training_pred["final_pred"] = final_valid_pred
valid_training_pred.to_csv(vpc.MODEL_DIR / "valid_training_predictions.csv", index=False)
normal_fold_report.to_csv(vpc.MODEL_DIR / "stage_a_normal_fold_report.csv", index=False)
dev_fold_report.to_csv(vpc.MODEL_DIR / "stage_b_deviation_fold_report.csv", index=False)

training_summary = pd.DataFrame([
    {"metric": "direct_valid_mean", "value": float(np.mean(direct_valid_clip))},
    {"metric": "two_stage_valid_mean", "value": float(np.mean(two_stage_valid_clip))},
    {"metric": "final_valid_mean", "value": float(np.mean(final_valid_pred))},
    {"metric": "n_model_features", "value": float(len(model_features))},
    {"metric": "normal_oof_mae", "value": float(mean_absolute_error(y_train, ts_train["two_stage_normal_pred"]))},
])
training_summary.to_csv(vpc.TRAINING_SUMMARY_PATH, index=False)

model_artifacts = {
    "model_features": model_features,
    "direct_models": final_models,
    "direct_cap_curve": direct_cap_curve,
    "normal_bank": normal_bank,
    "deviation_bank": deviation_bank,
    "dev_clip_low": dev_clip_low,
    "dev_clip_high": dev_clip_high,
    "final_cap_curve": final_cap_curve,
    "final_clip_config": final_clip_cfg,
    "final_two_stage_alpha": alpha,
    "constants": {
        "INSTALLED_CAPACITY_MW": INSTALLED_CAPACITY_MW,
        "HIGH_WIND_SPEED_COL": HIGH_WIND_SPEED_COL,
        "HIGH_WIND_START_WS": HIGH_WIND_START_WS,
        "HIGH_WIND_TRANSITION": HIGH_WIND_TRANSITION,
        "HIGH_WIND_CLIP_STRENGTH": HIGH_WIND_CLIP_STRENGTH,
        "TWO_STAGE_DEVIATION_SHRINK": TWO_STAGE_DEVIATION_SHRINK,
        "TWO_STAGE_DEVIATION_ABS_CLIP_MW": TWO_STAGE_DEVIATION_ABS_CLIP_MW,
        "TWO_STAGE_USE_PHYSICS_GATE": TWO_STAGE_USE_PHYSICS_GATE,
        "TWO_STAGE_GATE_MIN": TWO_STAGE_GATE_MIN,
        "TWO_STAGE_GATE_MAX": TWO_STAGE_GATE_MAX,
    },
}
joblib.dump(model_artifacts, vpc.MODEL_ARTIFACTS_PATH)

display(Markdown("### Training summary"))
display(training_summary)



Обучение ансамбля: saved_direct_ensemble
normal model: normal_hgb_l1
fold 1: MAE=9.149168, n_train=5385, n_valid=5405
fold 2: MAE=9.065562, n_train=10790, n_valid=5405
fold 3: MAE=8.943863, n_train=16195, n_valid=5405
fold 4: MAE=7.512626, n_train=21600, n_valid=5405
fold 5: MAE=7.856225, n_train=27005, n_valid=5405
OOF head rows filled by median: 26.465197475637734
normal model: normal_hgb_l2
fold 1: MAE=9.083945, n_train=5385, n_valid=5405
fold 2: MAE=9.118855, n_train=10790, n_valid=5405
fold 3: MAE=9.088999, n_train=16195, n_valid=5405
fold 4: MAE=7.664145, n_train=21600, n_valid=5405
fold 5: MAE=7.796613, n_train=27005, n_valid=5405
OOF head rows filled by median: 27.07558313759935
normal model: normal_hgb_q50
fold 1: MAE=9.004925, n_train=5385, n_valid=5405
fold 2: MAE=9.084428, n_train=10790, n_valid=5405
fold 3: MAE=8.944362, n_train=16195, n_valid=5405
fold 4: MAE=7.476703, n_train=21600, n_valid=5405
fold 5: MAE=7.848001, n_train=27005, n_valid=5405
OOF head rows filled by m

### Контрольный снимок конфигурации

Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска.

,group,parameter,value
0,paths,TRAIN_PATH,data\train_merged.csv
1,paths,VALID_PATH,data\valid_merged.csv
2,paths,TURBINE_COORDS_PATH,map\data\wind_farm_coords.csv
3,paths,OUT_DIR,outputs
4,paths,SUB_PATH,outputs\submission_final.csv
5,paths,SAVE_DIAGNOSTIC_ARTIFACTS,False
6,paths,SAVE_DIRECT_DEBUG_SUBMISSIONS,False
7,direct,BLEND_WEIGHTS_SUM,1.000001
8,direct,DIRECT_ENSEMBLE_FAST_MODE,False
9,physics,FULL_PHYSICS_BLOCK_ENABLED,True


N_LAYOUT_CLUSTERS: 4
WAKE_DIRECTION_STEP_DEG: 5
BEST_WAKE_FEATURE: layout_wake_risk_scalar_120m
ГЕОМЕТРИЯ ВЭС
Турбин: 26
Среднее ближайшее расстояние: 343.0 м
Медианное ближайшее расстояние: 353.6 м
Главная ось ВЭС: 9.30° / 189.30°
Доля объяснённой дисперсии PCA: [0.69031765 0.30968235]
BLEND_WEIGHTS sum: 1.000001
ENSEMBLE_PARAMS_FULL: {'cat_iter': 1200, 'xgb_estimators': 900, 'lgb_estimators': 900, 'hgb_iter': 650}
HGB_QUANTILES: [0.545, 0.57, 0.53]
FULL_PHYSICS_BLOCK_ENABLED: True
FULL_OOF_SPLITS: 5
FULL_CHANGEPOINT_WINDOW: 72
FULL_AVAIL_LAMBDA: 0.0015
TWO_STAGE_ENABLED: True
TWO_STAGE_FAST_MODE: False
TWO_STAGE_N_SPLITS: 5
FINAL_TWO_STAGE_ALPHA: 0.1
TS_DIR: outputs\ts


### Контрольный снимок конфигурации

Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска.

,group,parameter,value
0,paths,TRAIN_PATH,data\train_merged.csv
1,paths,VALID_PATH,data\valid_merged.csv
2,paths,TURBINE_COORDS_PATH,map\data\wind_farm_coords.csv
3,paths,OUT_DIR,outputs
4,paths,SUB_PATH,outputs\submission_final.csv
5,paths,SAVE_DIAGNOSTIC_ARTIFACTS,False
6,paths,SAVE_DIRECT_DEBUG_SUBMISSIONS,False
7,direct,BLEND_WEIGHTS_SUM,1.000001
8,direct,DIRECT_ENSEMBLE_FAST_MODE,False
9,physics,FULL_PHYSICS_BLOCK_ENABLED,True


N_LAYOUT_CLUSTERS: 4
WAKE_DIRECTION_STEP_DEG: 5
BEST_WAKE_FEATURE: layout_wake_risk_scalar_120m
ГЕОМЕТРИЯ ВЭС
Турбин: 26
Среднее ближайшее расстояние: 343.0 м
Медианное ближайшее расстояние: 353.6 м
Главная ось ВЭС: 9.30° / 189.30°
Доля объяснённой дисперсии PCA: [0.69031765 0.30968235]
BLEND_WEIGHTS sum: 1.000001
ENSEMBLE_PARAMS_FULL: {'cat_iter': 1200, 'xgb_estimators': 900, 'lgb_estimators': 900, 'hgb_iter': 650}
HGB_QUANTILES: [0.545, 0.57, 0.53]
FULL_PHYSICS_BLOCK_ENABLED: True
FULL_OOF_SPLITS: 5
FULL_CHANGEPOINT_WINDOW: 72
FULL_AVAIL_LAMBDA: 0.0015
TWO_STAGE_ENABLED: True
TWO_STAGE_FAST_MODE: False
TWO_STAGE_N_SPLITS: 5
FINAL_TWO_STAGE_ALPHA: 0.1
TS_DIR: outputs\ts


### Training summary

,metric,value
0,direct_valid_mean,39.724937
1,two_stage_valid_mean,41.164430
2,final_valid_mean,39.828905
3,n_model_features,185.000000
4,normal_oof_mae,10.671444


In [39]:
model_artifacts = joblib.load(vpc.MODEL_ARTIFACTS_PATH)
display(Markdown("### Состав сохраненных весов"))
display(pd.DataFrame([
    {"block": "direct_models", "items": len(model_artifacts["direct_models"]), "names": ", ".join(model_artifacts["direct_models"].keys())},
    {"block": "normal_bank", "items": len(model_artifacts["normal_bank"]["models"]), "names": ", ".join(model_artifacts["normal_bank"]["models"].keys())},
    {"block": "deviation_bank", "items": len(model_artifacts["deviation_bank"]["models"]), "names": ", ".join(model_artifacts["deviation_bank"]["models"].keys())},
    {"block": "model_features", "items": len(model_artifacts["model_features"]), "names": ""},
]))


### Состав сохраненных весов

,block,items,names
0,direct_models,6,"cat_mae_direct, xgb_residual, lgb_residual, hg..."
1,normal_bank,4,"normal_hgb_l1, normal_hgb_l2, normal_hgb_q50, ..."
2,deviation_bank,5,"dev_hgb_l1, dev_hgb_q50, dev_lgb_l1, dev_xgb_l..."
3,model_features,185,


In [40]:
valid_training_pred = pd.read_csv(vpc.MODEL_DIR / "valid_training_predictions.csv")
display(valid_training_pred.head(10))
plt.figure(figsize=(12, 4))
for col in ["direct_pred_clip", "two_stage_pred_clip", "final_pred"]:
    if col in valid_training_pred.columns:
        sns.kdeplot(valid_training_pred[col], label=col)
plt.title("Valid: распределение прогнозов после обучения")
plt.xlabel("МВт")
plt.legend()
plt.tight_layout()


,row_id,datetime,direct_pred_raw,direct_pred_clip,two_stage_pred_clip,final_pred
0,2125,2026-01-01 00:00:00,41.386208,41.386208,40.188873,41.266474
1,2124,2026-01-01 01:00:00,43.187956,43.187956,41.357876,43.004948
2,2123,2026-01-01 02:00:00,48.287468,48.287468,49.793261,48.438047
3,2122,2026-01-01 03:00:00,47.161992,47.161992,50.504474,47.496241
4,2121,2026-01-01 04:00:00,45.104565,45.104565,49.306116,45.524720
5,2120,2026-01-01 05:00:00,39.862360,39.862360,42.845721,40.160696
6,2119,2026-01-01 06:00:00,29.835571,29.835571,28.994329,29.751447
7,2118,2026-01-01 07:00:00,22.714891,22.714891,25.353796,22.978782
8,2117,2026-01-01 08:00:00,18.075245,18.075245,19.720150,18.239736
9,2116,2026-01-01 09:00:00,15.253585,15.253585,14.912150,15.219442
